In [31]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from pathlib import Path

np.random.seed(42)


In [32]:
data_dir = Path.cwd().parent /"data"/"raw"
train = pd.read_csv(data_dir/"train.csv")
dir_to_save = Path.cwd().parent /"data"/"processed"

In [33]:
feat_cols = [c for c in train.columns if c not in ["id", "anomaly"]]
train["channel"] = train["channel"].astype("category")

X = train[feat_cols]
y = train["anomaly"]
X_onehot = pd.get_dummies(X, columns=["channel"], drop_first=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [34]:
# --- Logistic Regression ---
oof_lr = np.zeros(len(train))
for tr_idx, val_idx in skf.split(X_onehot, y):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_onehot.iloc[tr_idx])
    Xval = scaler.transform(X_onehot.iloc[val_idx])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(Xtr, y.iloc[tr_idx])
    oof_lr[val_idx] = clf.predict_proba(Xval)[:, 1]
f1_lr = f1_score(y, (oof_lr > 0.5).astype(int))
print(f"Logistic Regression   F1 @ 0.5 threshold: {f1_lr:.4f}")


Logistic Regression   F1 @ 0.5 threshold: 0.8327


In [28]:
 # --- Random Forest ---
oof_rf = np.zeros(len(train))
for tr_idx, val_idx in skf.split(X_onehot, y):
    rf = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=3,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_onehot.iloc[tr_idx], y.iloc[tr_idx])
    oof_rf[val_idx] = rf.predict_proba(X_onehot.iloc[val_idx])[:, 1]
f1_rf = f1_score(y, (oof_rf > 0.5).astype(int))
print(f"Random Forest         F1 @ 0.5 threshold: {f1_rf:.4f}")


Random Forest         F1 @ 0.5 threshold: 0.8743


In [29]:
# --- LightGBM ---
oof_lgb = np.zeros(len(train))
for tr_idx, val_idx in skf.split(X, y):
    model = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.03, num_leaves=15,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
        class_weight="balanced", random_state=42, verbosity=-1,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx], categorical_feature=["channel"])
    oof_lgb[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
f1_lgb = f1_score(y, (oof_lgb > 0.5).astype(int))
print(f"LightGBM              F1 @ 0.5 threshold: {f1_lgb:.4f}")

LightGBM              F1 @ 0.5 threshold: 0.9016


In [30]:
# save out-of-fold probabilities for the next step (threshold tuning)
np.save(dir_to_save/"oof_lr.npy", oof_lr)
np.save(dir_to_save/"oof_rf.npy", oof_rf)
np.save(dir_to_save/"oof_lgb.npy", oof_lgb)
np.save(dir_to_save/"y_train.npy", y.values)
print("\nSaved OOF probabilities for 04_threshold_tuning.py")


Saved OOF probabilities for 04_threshold_tuning.py
